# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [1]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

Loaded: 1704 rows
       country continent  year  lifeExp       pop   gdpPercap iso_alpha  \
0  Afghanistan      Asia  1952   28.801   8425333  779.445314       AFG   
1  Afghanistan      Asia  1957   30.332   9240934  820.853030       AFG   
2  Afghanistan      Asia  1962   31.997  10267083  853.100710       AFG   
3  Afghanistan      Asia  1967   34.020  11537966  836.197138       AFG   
4  Afghanistan      Asia  1972   36.088  13079460  739.981106       AFG   

   iso_num  
0        4  
1        4  
2        4  
3        4  
4        4  


In [ ]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1704 entries, 0 to 1703
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   country    1704 non-null   object 
 1   continent  1704 non-null   object 
 2   year       1704 non-null   int64  
 3   lifeExp    1704 non-null   float64
 4   pop        1704 non-null   int64  
 5   gdpPercap  1704 non-null   float64
 6   iso_alpha  1704 non-null   object 
 7   iso_num    1704 non-null   int64  
dtypes: float64(2), int64(3), object(3)
memory usage: 106.6+ KB
None
Years: [1952, 1957, 1962, 1967, 1972, 1977, 1982, 1987, 1992, 1997, 2002, 2007]
Continents: ['Asia' 'Europe' 'Africa' 'Americas' 'Oceania']
         year  lifeExp           pop  gdpPercap  iso_num
count  1704.0   1704.0  1.704000e+03     1704.0   1704.0
mean   1979.5     59.5  2.960121e+07     7215.3    425.9
std      17.3     12.9  1.061579e+08     9857.5    248.3
min    1952.0     23.6  6.001100e+04      241.2      4.0


## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [ ]:
# Task 1 — Asia: 2002 vs 2007
# ---------------------------
import math
import plotly.graph_objects as go

continent = 'Asia'
sub = df.loc[(df['continent'] == continent) & (df['year'].isin([2002, 2007]))].copy()
sub['year'] = sub['year'].astype(str)          


wide = sub.pivot(index='country', columns='year', values=['lifeExp', 'gdpPercap'])
gain = wide['lifeExp']['2007'] - wide['lifeExp']['2002']
n_longer = (gain > 0).sum()
n_richer = (wide['gdpPercap']['2007'] > wide['gdpPercap']['2002']).sum()

fig = px.scatter(
    data_frame=sub, x='gdpPercap', y='lifeExp',
    color='year',
    color_discrete_map={'2002': 'lightgrey', '2007': 'crimson'},   
    hover_name='country',
    hover_data={'year': True},
    log_x=True,                                                 
    labels={'gdpPercap': 'GDP per Capita (USD, log scale)',
            'lifeExp': 'Life Expectancy', 'year': ''},
    template='plotly_white',
    title=('Asia moved up and to the right: all {n_longer} countries live longer in 2007'),
    height=550
)
fig.update_traces(marker=dict(size=9, opacity=0.85))

for country, g in sub.sort_values('year').groupby('country'):
    fig.add_trace(go.Scatter(x=g['gdpPercap'], y=g['lifeExp'], mode='lines',
                             line=dict(color='lightgrey', width=1),
                             hoverinfo='skip', showlegend=False))
fig.data = fig.data[2:] + fig.data[:2]         

best = gain.idxmax()
low = sub.loc[sub['year'] == '2007'].nsmallest(1, 'lifeExp').iloc[0]
for name, text, ax, ay in [
    (best, f'{best}: +{gain[best]:.1f} yrs<br>(biggest gain)', -70, -45),
    (low['country'], f"{low['country']}: lowest life<br>expectancy in Asia", 80, 0),
]:
    row = sub.loc[(sub['country'] == name) & (sub['year'] == '2007')].iloc[0]
    fig.add_annotation(
        x=math.log10(row['gdpPercap']), y=row['lifeExp'],
        text=text, showarrow=True, arrowhead=1, arrowcolor='crimson', ax=ax, ay=ay,
        font=dict(size=11, family='Arial', color='crimson'), align='left'
    )

fig.update_layout(
    font=dict(family='Arial', size=13),
    xaxis=dict(showgrid=False),
    yaxis=dict(gridcolor='#EEEEEE'),
    legend=dict(orientation='h', x=0, y=1.05, title='')
)
fig.show()


SyntaxError: unterminated string literal (detected at line 26) (483767686.py, line 26)

### Explanation — Task 1
**Design choices (lecture principles):**
- **Pre-attentive attribute = colour.** 2002 is `lightgrey` (background/context), 2007 is `crimson` (the message). The eye goes to the red "now" points first → visual hierarchy: present state first, past as context.
- **Only 2 colours**, because colour encodes one categorical variable (year) — no rainbow.
- **Log x-axis**: GDP spans ~$900 to ~$47,000; on a linear scale most countries would overplot on the left.
- **Connector lines** show the direction of change (grey → red, mostly up and to the right) without extra legend reading — less load on short-term memory.
- **Direct labels only for notable points** (biggest gain and the low outlier), not for every country.
- **Insight title** states the takeaway, computed from the data.

**What each command does:**
- `df.loc[(continent == 'Asia') & year.isin([2002, 2007])]` — filter to one continent and two years.
- `astype(str)` — makes year categorical, so `color='year'` gives 2 discrete colours instead of a colour scale.
- `pivot(...)` — one row per country with 2002 and 2007 side by side → compute gain and counts for the title.
- `px.scatter(..., color_discrete_map=..., log_x=True, hover_name='country')` — the scatter with fixed colours, log axis and country name on hover.
- `go.Scatter(mode='lines')` in the loop — draws each country's 2002→2007 connector; `hoverinfo='skip'` keeps hover clean.
- `fig.data = fig.data[2:] + fig.data[:2]` — reorders traces so lines sit under the dots.
- `gain.idxmax()` / `nsmallest(1, 'lifeExp')` — find the two countries worth labelling.
- `add_annotation(x=math.log10(...))` — on a log axis Plotly expects annotation x in log10; `ax`/`ay` offset the label in pixels with an arrow.
- `update_layout(...)` — Arial font, light horizontal grid only, legend in one row without a title.


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [ ]:
# Task 2 — Bubble chart, 2007: Asia highlighted
# ----------------------------------------------
import math

d07 = df.loc[df['year'] == 2007].copy()
highlight_continent = 'Asia'
colour_map = {c: ('crimson' if c == highlight_continent else 'lightgrey') for c in d07['continent'].unique()}

asia_share = d07.loc[d07['continent'] == highlight_continent, 'pop'].sum() / d07['pop'].sum()
world_avg = d07['lifeExp'].mean()

fig = px.scatter(
    data_frame=d07, x='gdpPercap', y='lifeExp',
    size='pop', size_max=60,                    # cap bubble size so small countries stay visible
    color='continent',
    color_discrete_map=colour_map,
    category_orders={'continent': [c for c in d07['continent'].unique() if c != highlight_continent]
                                  + [highlight_continent]},   # Asia drawn last = on top
    hover_name='country',
    custom_data=['country', 'pop'],
    log_x=True,
    opacity=0.7,                                # reduces overplotting
    labels={'gdpPercap': 'GDP per Capita (USD, log scale)', 'lifeExp': 'Life Expectancy',
            'pop': 'Population', 'continent': ''},
    template='plotly_white',
    title=f'Asia holds {asia_share:.0%} of the world population — most of it in two mid-income giants, China and India',
    height=600
)

# Formatted hover
fig.update_traces(
    hovertemplate='<b>%{hovertext}</b><br>'
                  'GDP per capita: $%{x:,.0f}<br>'
                  'Life expectancy: %{y:.1f} yrs<br>'
                  'Population: %{customdata[1]:,.0f}'
)

# Reference line: world average life expectancy (context)
fig.add_hline(y=world_avg, line_dash='dot', line_color='grey', line_width=1,
              annotation_text=f'World average: {world_avg:.0f} yrs',
              annotation_position='bottom right',
              annotation_font=dict(size=11, color='grey'))

# Direct labels on the two bubbles that drive the story
for country, ax, ay in [('China', 120, -70), ('India', 120, 75)]:
    row = d07[d07['country'] == country].iloc[0]
    fig.add_annotation(
        x=math.log10(row['gdpPercap']), y=row['lifeExp'],
        text=f"<b>{country}</b>: {row['pop']/1e9:.2f} bn people",
        showarrow=True, arrowhead=1, arrowcolor='crimson', ax=ax, ay=ay,
        font=dict(size=12, family='Arial', color='crimson'), bgcolor='white'
    )

# Story box (replaces the legend)
fig.add_annotation(
    x=math.log10(400), y=80,
    text='<b>Asia</b> in red<br>bubble size = population',
    showarrow=False, align='left',
    font=dict(color='crimson', size=11, family='Arial'),
    bgcolor='white', bordercolor='crimson', borderwidth=1, borderpad=4
)

fig.update_layout(
    font=dict(family='Arial', size=13),
    xaxis=dict(showgrid=False),
    yaxis=dict(gridcolor='#EEEEEE'),
    showlegend=False
)
fig.show()


### Explanation — Task 2
**Design choices (lecture principles):**
- **Grey + highlight**: Asia in `crimson`, every other continent `lightgrey`. Colour is the pre-attentive attribute that tells the viewer where to look within ~250 ms (iconic memory); grey pushes the rest into the background.
- **Size = third numerical variable** (population), with `size_max=60` so China/India don't swallow the chart.
- **Log x-axis** + `opacity=0.7` against overplotting.
- **Reference line** (world average life expectancy) gives context: China sits above it, India just below.
- **Direct labels** on China and India only (the notable points), and a small text box instead of a legend — fewer things to hold in short-term memory.
- **Insight title** combines the visual with a clear takeaway → helps the message reach long-term memory.

**What each command does:**
- `df.loc[df['year'] == 2007]` — all countries, one year.
- `colour_map = {c: 'crimson' if c == highlight_continent else 'lightgrey' ...}` — builds the grey + highlight colour map, same pattern as Example 3 in the notes.
- `asia_share`, `world_avg` — numbers for the title and the reference line.
- `px.scatter(..., size='pop', size_max=60)` — turns the scatter into a bubble chart; `size_max` is the largest bubble in pixels.
- `category_orders` — puts Asia last so its bubbles are drawn on top of the grey ones.
- `custom_data=['country', 'pop']` + `hovertemplate` — custom hover text with formatted GDP, life expectancy and population (`%{customdata[1]:,.0f}` adds thousand separators).
- `fig.add_hline(...)` — dotted horizontal line at the world average with a small label.
- `add_annotation(x=math.log10(...), ax=..., ay=...)` — arrow labels for China and India; log10 because the x-axis is log-scaled.
- Last `add_annotation(showarrow=False, bordercolor=...)` — the bordered story box, like in Example 3.
- `showlegend=False` — legend removed; the box and labels already explain the colours.
